# 📑 CoC Pool Data
 
## 🔗 Links
- **[BRD CoC Pool Data](https://cepalhellas.atlassian.net/wiki/spaces/DBTC/pages/933429250/CoCPoolData+BRD+for+Cloud+Migration+and+UAT+Summary)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket](https://cepalhellas.atlassian.net/browse/ASK-4873?search_id=55cb4a07-b8a6-46ec-8800-be91d8885f8f)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket](https://cepalhellas.atlassian.net/browse/ASK-4873?search_id=b80938d3-82fb-43db-9640-13e8a3409f31)**  
  A summary of findings from UAT.
 
---


In [0]:
import os
import sys

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/CoCPoolData/", 1)[0]
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname

env = get_env_prefix()

In [0]:
# Reads portfolio configs from config/live/*.json and outputs them as a JSON array
# for use by a for_each_task downstream.

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("portfolio", "")  # comma-separated filter, empty = all
dbutils.widgets.text("config_folder", "live")

import os
import sys
import json

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/DetailedPortfolio/", 1)[0]
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs

In [0]:
import json
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
try:    
    config_folder = dbutils.widgets.get("config_folder")
except Exception as e:
    raise ValueError("Missing required input: config_folder")
try:    
    portfolio = dbutils.widgets.get("portfolio")
except Exception as e:
    raise ValueError("Missing required input: portfolio")

run_date_raw = dbutils.widgets.get("run_date").strip()
if run_date_raw:
    run_date = run_date_raw
else:
    run_date = spark.sql(f"SELECT MAX(BusinessDate) FROM `{env}-cdh`.primaryactive.exposure").collect()[0][0].strftime("%Y-%m-%d")
run_date_str = run_date.replace('-', '')
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

report_name = 'CocPoolData'
filter_set = {p.strip().lower() for p in portfolio.split(",") if p.strip()} if portfolio else None
config = load_configs(dbutils, config_folder, filter_set, report_name="CoCPoolData")[0]
report_args = config["report_args"].copy()
# portfolio_name = report_args["portfolio_name"]
print(f"Report {report_name} is started")

# Read JSON
# json_data = json.loads(report_args)
# Access the values and store them into variables
if "portfolio_name" not in report_args: raise KeyError("'portfolio_name' key is missing in the report_args JSON data.")
if "output_file_prefix" not in report_args: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in report_args: raise KeyError("'report_output' key is missing in the report_args JSON data.")

portfolioName = "".join(report_args["portfolio_name"])
output_file_prefix = "".join(report_args["output_file_prefix"])
report_output = "".join(report_args["report_output"])
# Resolve full export path from SSM if relative (same pattern as MegaDataTape)
if not report_output.startswith("/Volumes"):
    from aws_utils import get_scope_key_names, get_aws_credentials, connect_to_aws_and_access_ssm
    import boto3
    _secrets_scope = "aws-databricks"
    _aws_keys = get_scope_key_names(_secrets_scope)
    _aws_key = _aws_keys[0] if _aws_keys else None
    _aws_secret_key = get_aws_credentials(_secrets_scope, _aws_key)
    _ssm_client = boto3.client("ssm", region_name="eu-central-1",
                               aws_access_key_id=_aws_key,
                               aws_secret_access_key=_aws_secret_key)
    export_base_path = connect_to_aws_and_access_ssm(
        "/reports/exports/s3_main_path", _ssm_client, _aws_secret_key
    ).replace("envprefix", env)
    report_output = export_base_path + report_output
print(f"Report {report_name} is running with the following arguments: report_args = {report_args}, report_name = {report_name}, portfolio_name = {portfolioName}, output_file_prefix = {output_file_prefix}, run_date = {run_date}")
print(f"now is: {now_datetime_str}")

In [0]:
# --- Parameters ---
portfolio_filter = "'Mercury','Orbit','Light','Cell','Pearl Portfolio A'"

# Exposure — qca source, CoCLetter proxy filter applied, party types filtered
exposure = spark.sql(f"""
       SELECT ExposureKey, AgreementKey, CDI, PartyName, Currency,
              PortfolioName, PortfolioKey, SourceSystem,
              PartyRelationshipType,
              OutstandingTotalBalance, OutstandingTotalBalanceMLT,
              DuePrincipal, Dueinterest, Penaltyinterest,
              DueUnrealizedinterest, PenaltyUnrealizedinterest,
              DueExpense, DueCommission, DueAccrued,
              DueAmount,
              DaysPastDueWithMateriality, BucketCode, Bucket,
              AssetSegment, AssetSegmentCode,
              ExposureStatus, ExposureStatusCode,
              Denounceddate, AdministrationStatus, AdministrationStatusCode,
              LetterStatus,
              FirstLetterDate, FirstLetterFlag,
              FirstLetterReminderDate, FirstLetterReminderFlag,
              SecondLetterDate, SecondLetterFlag,
              ThirdLetterDate, ThirdLetterFlag,
              CoCAccountStatusDate, CoCStatus,
              ExclusionReason, ExclusionDate,
              ProductCode, ProductKey,
              PartyTaxRegistrationNumber,
              BusinessDate
       FROM `p-cdh`.primary.get_all_exposures(DATE({run_date!r}))
       WHERE PortfolioName IN ({portfolio_filter})
         AND SourceSystem = 'qca'
         AND PartyRelationshipType IN ('Primary Owner', 'Co-Owner', 'Guarantor')
         AND (
             LetterStatus IS NOT NULL
             OR NOT (
                 COALESCE(CAST(FirstLetterDate AS STRING), '1753-01-01') IN ('1753-01-01', '1900-01-01')
                 AND COALESCE(CAST(SecondLetterDate AS STRING), '1753-01-01') IN ('1753-01-01', '1900-01-01')
                 AND COALESCE(CAST(ThirdLetterDate AS STRING), '1753-01-01') IN ('1753-01-01', '1900-01-01')
             )
             OR ExclusionReason IS NOT NULL
         )
""")
exposure.createOrReplaceTempView("exposure")


# Derivative Exposure — EverDenouncedFlag, compound ProductKey, DueAmount, CaseCode bridge
dexposure = spark.sql(f"""
       SELECT DISTINCT ExposureKey,
              EverDenouncedFlag,
              ProductKey AS DxProductKey,
              DueAmount AS DxDueAmount,
              CAST(CaseCode AS STRING) AS DxCaseCode,
              LPS, LPSEng
       FROM `{env}-cdh`.derivativeactive.dexposure
       WHERE BusinessDate = {run_date!r}
         AND PortfolioName IN ({portfolio_filter})
""")
dexposure.createOrReplaceTempView("dexposure")


# Party — qca source, includes Segment and PartyKey (used for cust_code)
party = spark.sql(f"""
       SELECT DISTINCT CDI, PartyType, PartyID, PartyKey,
              FirstNameGR, LastNameGR, FirstFatherNameGR,
              DeceasedFlag, L3869Status,
              Segment, SegmentCode,
              PortfolioName
       FROM `{env}-cdh`.primaryactive.party
       WHERE BusinessDate = {run_date!r}
         AND SourceSystem = 'qca'
""")
party.createOrReplaceTempView("party")


# Casecode — bridged via dexposure.DxCaseCode (no ExposureKey/CDI on this table)
casecode = spark.sql(f"""
       SELECT DISTINCT CaseCodeKey, CaseCodePrimary,
              CustCode, DebtCode,
              Queue, QueueCode,
              AdminStatus, AdminStatusCode,
              CurrentlyOpenFlag,
              DecisionDate, CommitteeDecisionType,
              ApplicationDecisionEverDate,
              ActiveAppealFlag,
              CaseRelationTypeCode,
              PortfolioName, SourceSystem
       FROM `{env}-cdh`.primaryactive.casecode
       WHERE BusinessDate = {run_date!r}
         AND SourceSystem = 'qca'
""")
casecode.createOrReplaceTempView("casecode")

# LPS Agreement Debtor — operational legal status at agreement+debtor level
# Source: derivativeactive (same CDH layer as dexposure)
# Join keys: AgreementKey → AgreementKey, CDI → CDI (direct match, no prefix stripping)
# 1:1 per AgreementKey+CDI (zero fan-out verified). cc.LPSStatus is 100% NULL — this is the real LPS source
LPSStatus = spark.sql(f"""
       SELECT DISTINCT 
              AgreementKey AS LpsAgreementKey,
              CDI AS LpsCDI,
              OperationalLegalStatusLevel3
       FROM `{env}-cdh`.derivativeactive.dlegalprocessstatusagreementdebtor
       WHERE BusinessDate = {run_date!r}
""")
LPSStatus.createOrReplaceTempView("LPSStatus")


# Assignment — deduplicated to 1 row per ExposureKey, ProcessType only
assignment = spark.sql(f"""
       SELECT ExposureKey,
              MAX(ProcessType) AS ProcessType
       FROM `{env}-cdh`.primaryactive.assignment
       WHERE BusinessDate = {run_date!r}
       GROUP BY ExposureKey
""")
assignment.createOrReplaceTempView("assignment")


# Product — join key is compound 'ProductCode;SubProductCode' via dexposure.DxProductKey
product = spark.sql(f"""
       SELECT DISTINCT 
              CONCAT(ProductCode, ';', CAST(SubProductCode AS STRING)) AS ProductKeyCompound,
              ProductCode, ProductDescription,
              SubProductCode, SubProduct
       FROM `{env}-cdh`.primaryactive.product
       WHERE BusinessDate = {run_date!r}
""")
product.createOrReplaceTempView("product")


# Contact Details — CommunicationFlag is STRING type ('0','1','1019','1020')
contactdetails = spark.sql(f"""
       SELECT DISTINCT CDI, PartyKey,
              Street, StreetNumber, PostalCode, City,
              ContactDetailType, ContactDetailTypeCode,
              CommunicationFlag
       FROM `{env}-cdh`.primaryactive.contactdetails
       WHERE BusinessDate = {run_date!r}
""")
contactdetails.createOrReplaceTempView("contactdetails")


# Under Law — used for Ένταξη σε Νόμο and OCW status. L4738 rows have CDI=NULL.
underlaw = spark.sql(f"""
       SELECT DISTINCT CDI, TaxRegistrationNumber,
              PlatformName, InLawFlag,
              UnderLawType, UnderLawTypeCode,
              ApplicationStatus, ApplicationStatusDate,
              SubmissionDate,
              PortfolioName
       FROM `{env}-cdh`.primaryactive.underlaw
       WHERE BusinessDate = {run_date!r}
""")
underlaw.createOrReplaceTempView("underlaw")


# Settlement Exposure — IsActiveFlag=1 for active settlements
settlementexposure = spark.sql(f"""
       SELECT DISTINCT ExposureKey,
              SettlementStatus, IsActiveFlag
       FROM `{env}-cdh`.primaryactive.settlementexposure
       WHERE BusinessDate = {run_date!r}
""")
settlementexposure.createOrReplaceTempView("settlementexposure")


# RePoint CoC active cases — open cases with an active C-code stage
# Letter_(Conduct_Code) mapping (from cocpooldata_onprem_2025_03_16 join):
#   7978=C1, 7979=C2, 7980=C3, 7981=C4, 7982=C6  (C5 had 0 rows in historical data)
# 'C6 Sent' = 3rd letter confirmed dispatched → cycle complete = QC 8004 equiv. (EXCLUDED)
# 'C6'      = 3rd letter in-flight/pending    → active         = QC 7982 equiv.  (INCLUDED)
# 'C1 Sent'..'C5 Sent' = intermediate confirmed stages, case still open           (INCLUDED)
# Loaded daily (Mon-Fri); Ref_Date lags ~4 days — acceptable as transitions are weekly.
repoint_coc = spark.sql(f"""
       SELECT DISTINCT CAST(Case_Code AS INT) AS repoint_case_code
       FROM `{env}-cdh`.landed.repointqcab_coc_fields
       WHERE BusinessDate = {run_date!r}
         AND Case_Status = 'Open'
         AND `Letter_(Conduct_Code)` IN ('C1 Sent', 'C2 Sent', 'C3 Sent', 'C4 Sent', 'C5 Sent', 'C6')
""")
repoint_coc.createOrReplaceTempView("repoint_coc")

In [0]:
# --- CTE equivalents ---

# Primary address (CommunicationFlag = '1')
add_primary = (contactdetails
    .filter(F.col("CommunicationFlag") == "1")
    .select("CDI",
            F.col("Street").alias("Street1"),
            "PostalCode", "City",
            F.col("ContactDetailType").alias("AddressType"))
)

# Other addresses: dedup within each type, then rank across types
add_rest_ranked = (contactdetails
    .filter((F.col("CommunicationFlag") != "1") & F.col("CommunicationFlag").isNotNull())
    .withColumn("dedup", F.row_number().over(
        Window.partitionBy("CDI", "ContactDetailTypeCode").orderBy("ContactDetailTypeCode")))
    .filter(F.col("dedup") == 1)
    .withColumn("rnk", F.row_number().over(
        Window.partitionBy("CDI").orderBy(F.col("ContactDetailTypeCode").asc())))
    .select("CDI", "Street", "rnk")
)

s2 = add_rest_ranked.filter(F.col("rnk") == 1).select("CDI", "Street")
s3 = add_rest_ranked.filter(F.col("rnk") == 2).select("CDI", "Street")
s4 = add_rest_ranked.filter(F.col("rnk") == 3).select("CDI", "Street")
s5 = add_rest_ranked.filter(F.col("rnk") == 4).select("CDI", "Street")

# Under Law: split CDI/TaxReg joins
ul_cdi_df = (underlaw
    .filter(F.col("PlatformName").isNotNull() & (F.col("PlatformName") != "N/A") & F.col("CDI").isNotNull())
    .select("CDI", "PlatformName")
)
ul_tax_df = (underlaw
    .filter(F.col("PlatformName").isNotNull() & (F.col("PlatformName") != "N/A") &
            F.col("CDI").isNull() & F.col("TaxRegistrationNumber").isNotNull())
    .select("TaxRegistrationNumber", "PlatformName")
)

# OCW status: deduplicated to 1 row per CDI / TaxReg (latest status)
ocw_cdi_df = (underlaw
    .filter((F.col("UnderLawType") == "N4738/2020") & F.col("CDI").isNotNull())
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("CDI").orderBy(F.col("ApplicationStatusDate").desc())))
    .filter(F.col("_rn") == 1)
    .select("CDI",
            F.col("ApplicationStatus").alias("OCWStatus"),
            F.col("SubmissionDate").alias("OCWSubmittedDate"),
            F.col("ApplicationStatusDate").alias("OCWStatusDate"))
)
ocw_tax_df = (underlaw
    .filter((F.col("UnderLawType") == "N4738/2020") &
            F.col("CDI").isNull() & F.col("TaxRegistrationNumber").isNotNull())
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("TaxRegistrationNumber").orderBy(F.col("ApplicationStatusDate").desc())))
    .filter(F.col("_rn") == 1)
    .select("TaxRegistrationNumber",
            F.col("ApplicationStatus").alias("OCWStatus"),
            F.col("SubmissionDate").alias("OCWSubmittedDate"),
            F.col("ApplicationStatusDate").alias("OCWStatusDate"))
)

# Active settlements per ExposureKey
sa_df = (settlementexposure
    .filter(F.col("IsActiveFlag") == 1)
    .groupBy("ExposureKey")
    .agg(F.max("SettlementStatus").alias("SettlementStatus"))
)


# --- Build main joined DataFrame ---
df = (exposure.alias("e")
    .join(dexposure.alias("dx"),
          F.regexp_replace(F.col("e.ExposureKey"), r'^[A-Za-z]+', '') ==
          F.regexp_replace(F.col("dx.ExposureKey"), r'^[A-Za-z]+', ''), "left")
    .join(party.alias("p"),
          (F.col("p.CDI") == F.col("e.CDI")) & (F.col("p.PortfolioName") == F.col("e.PortfolioName")), "left")
    .join(assignment.alias("asn"),
          F.regexp_replace(F.col("asn.ExposureKey"), r'^[A-Za-z]+', '') ==
          F.regexp_replace(F.col("e.ExposureKey"), r'^[A-Za-z]+', ''), "left")
    .join(casecode.alias("cc"),
          F.col("cc.CaseCodeKey") == F.col("dx.DxCaseCode"), "left")
    .join(product.alias("pr"),
          F.col("pr.ProductKeyCompound") == F.col("dx.DxProductKey"), "left")
    .join(add_primary.alias("ad"),
          F.col("ad.CDI") == F.col("e.CDI"), "left")
    .join(s2.alias("s2"), F.col("s2.CDI") == F.col("e.CDI"), "left")
    .join(s3.alias("s3"), F.col("s3.CDI") == F.col("e.CDI"), "left")
    .join(s4.alias("s4"), F.col("s4.CDI") == F.col("e.CDI"), "left")
    .join(s5.alias("s5"), F.col("s5.CDI") == F.col("e.CDI"), "left")
    .join(ul_cdi_df.alias("ul_cdi"),
          F.col("ul_cdi.CDI") == F.col("e.CDI"), "left")
    .join(ul_tax_df.alias("ul_tax"),
          F.col("ul_tax.TaxRegistrationNumber") == F.col("e.PartyTaxRegistrationNumber"), "left")
    .join(ocw_cdi_df.alias("ocw_cdi"),
          F.col("ocw_cdi.CDI") == F.col("e.CDI"), "left")
    .join(ocw_tax_df.alias("ocw_tax"),
          F.col("ocw_tax.TaxRegistrationNumber") == F.col("e.PartyTaxRegistrationNumber"), "left")
    .join(sa_df.alias("sa"),
          F.regexp_replace(F.col("sa.ExposureKey"), r'^[A-Za-z]+', '') ==
          F.regexp_replace(F.col("e.ExposureKey"), r'^[A-Za-z]+', ''), "left")
    .join(LPSStatus.alias("lps"),
          (F.col("e.AgreementKey") == F.col("lps.LpsAgreementKey")) &
          (F.col("lps.LpsCDI") == F.col("e.CDI")), "left")
)


# --- SELECT with column transformations ---
df_cocpool = df.select(

    # Identification
    F.col("e.LetterStatus").alias("Letter_Type"),
    F.col("e.AssetSegment").alias("Type_Of_Issuing"),  
    F.col("e.PortfolioName"),
    F.col("e.AgreementKey").alias("Contract_Number"),
    F.col("e.ExposureKey").alias("Account_Number"),
    F.col("e.Currency").alias("Case_Currency"),
    F.col("e.CDI").alias("CIF"),
    F.col("p.PartyKey").alias("cust_code"),
    F.col("cc.CaseCodePrimary").alias("CaseCode"),
    F.col("e.PartyName").alias("Party_Name"),

    # Financial
    F.coalesce(F.col("e.OutstandingTotalBalance").cast("decimal(21,2)"), F.lit(0)).alias("Outstanding_Balance_LCY"),
    F.coalesce(F.col("e.OutstandingTotalBalanceMLT").cast("decimal(21,2)"), F.lit(0)).alias("MLT_Amount_LCY"), 
    (F.coalesce(F.col("e.OutstandingTotalBalanceMLT").cast("decimal(21,2)"), F.lit(0)) +
     F.coalesce(F.col("e.OutstandingTotalBalance").cast("decimal(21,2)"), F.lit(0))).alias("CoC_Outstanding_Balance_LCY"),
    F.coalesce(F.col("dx.DxDueAmount").cast("decimal(28,2)"),
               F.col("e.DueAmount").cast("decimal(28,2)"), F.lit(0)).alias("Debt_Amount_LCY"),

    # Application number (substring before first '-')
    F.when(F.locate("-", F.col("e.AgreementKey")) > 0,
           F.col("e.AgreementKey").substr(F.lit(1), F.locate("-", F.col("e.AgreementKey")) - 1)
    ).otherwise(F.col("e.AgreementKey")).alias("Application_Number"),

    F.col("e.PartyTaxRegistrationNumber").alias("AFM"),
    F.col("e.DaysPastDueWithMateriality").alias("Deliquency_Days"),

    # Case & Legal
    F.col("cc.Queue"),
    F.col("e.ExposureStatus").alias("Case_Denounce_Status"),
    F.col("asn.ProcessType").alias("Packet_Process_Type"),
    F.col("lps.OperationalLegalStatusLevel3").alias("Legal_Process_Status"),   
    F.col("dx.LPSEng").alias("Debt_Legal_Process_Status"),

    # Product
    F.col("pr.ProductDescription").alias("ProductDesc"),
    F.col("pr.SubProductCode").alias("Subproduct_Code"),
    F.col("pr.SubProduct").alias("Subproduct"),

    F.col("e.BucketCode").alias("Current_Bucket"),
    F.coalesce(F.col("ul_cdi.PlatformName"), F.col("ul_tax.PlatformName")).alias("Entaxi_se_Nomo"),
    F.col("p.PartyType").alias("Legal_Entity_Indication"),

    # CoC Letter fields
    F.col("e.FirstLetterDate").alias("Account_First_Letter_Date"),
    F.col("e.FirstLetterFlag").alias("First_CoC_Letter_Date_Flag"),
    F.col("e.FirstLetterReminderDate").alias("Account_First_Reminder_Letter_Date"),
    F.col("e.FirstLetterReminderFlag").alias("First_CoC_Letter_Reminder_Date_Flag"),
    F.col("e.SecondLetterDate").alias("Account_Second_Letter_Date"),
    F.col("e.SecondLetterFlag").alias("Second_CoC_Letter_Date_Flag"),
    F.col("e.ThirdLetterDate").alias("Account_Third_Letter_Date"),
    F.col("e.ThirdLetterFlag").alias("Third_CoC_Letter_Date_Flag"),
    F.col("e.CoCAccountStatusDate").alias("Account_CoC_Letter_Date"),

    # Committee
    F.col("cc.DecisionDate").alias("Committee_Decision_Date"),
    F.col("cc.CommitteeDecisionType").alias("Committee_Decision"),

    # Wholesale (portfolio + segment logic)
    F.when(F.col("e.PortfolioName").isin("Orbit", "Mercury"), "NO")
     .when((F.col("e.PortfolioName") == "Light") & (F.col("p.Segment") == "Wholesale_CP"), "YES")
     .when((F.col("e.PortfolioName") == "Light") & (F.coalesce(F.col("p.Segment"), F.lit("")) != "Wholesale_CP"), "NO")
     .otherwise("").alias("Wholesale_Account"),

    F.col("p.L3869Status").alias("Customer_N_3869"),

    F.when(F.col("p.Segment").isNotNull() & (F.col("p.Segment") != "") & (F.col("p.Segment") != "N/A"),
           F.col("p.Segment")).otherwise(F.lit(None).cast("string")).alias("Account_Segment"),

    F.col("e.CoCStatus").alias("DEK_Account_Status"),

    # Settlement
    F.when(F.col("sa.ExposureKey").isNotNull(), 1).otherwise(0).alias("Has_Settlement_Active"),
    F.col("sa.SettlementStatus"),

    F.col("dx.EverDenouncedFlag").cast("string").alias("EverDenouncedFlag"),
    F.col("e.Denounceddate").alias("DenouncedDate"),
    F.col("e.AdministrationStatusCode").alias("Debt_Admin_Status"),
    F.col("cc.ApplicationDecisionEverDate").alias("ApplicationCodeEverCompletedDate"),
    F.lit(None).cast("string").alias("ObjectionDate"),
    F.col("e.ExclusionReason").alias("Exclusion_Reason_from_CoC_Letters"),
    F.col("e.ExclusionDate").alias("ExclusionDate"),
    F.col("e.PartyRelationshipType").alias("case_relation_type"),
    F.col("cc.CurrentlyOpenFlag"),
    F.col("p.PartyKey").alias("PartyId"),
    F.lit(None).cast("string").alias("Legal_CIF_Status_CoC"),
    F.col("cc.AdminStatus").alias("Admin_Status"),

    # Party details
    F.col("p.FirstNameGR").alias("First_Name"),
    F.col("p.LastNameGR").alias("Last_Name"),
    F.col("p.FirstFatherNameGR").alias("Father_Name"),

    # Address
    F.col("ad.Street1").alias("Street_1"),
    F.col("s2.Street").alias("Street_2"),
    F.col("s3.Street").alias("Street_3"),
    F.col("s4.Street").alias("Street_4"),
    F.col("s5.Street").alias("Street_5"),
    F.col("ad.PostalCode").alias("Postal_Code"),
    F.col("ad.City"),
    F.col("ad.AddressType").alias("Address_Type"),
    F.lit(None).cast("string").alias("Street_Number"),

    # Account Status: Greek to English mapping
    F.when(F.col("e.ExposureStatus") == "Χωρίς Καθυστέρηση", "Normal")
     .when(F.col("e.ExposureStatus") == "Χωρις Καθυστέρηση", "Normal")
     .when(F.col("e.ExposureStatus") == "Σε καθυστέρηση", "Overdue")
     .when(F.col("e.ExposureStatus") == "Καταγγελία", "Define Delay")
     .when(F.col("e.ExposureStatus") == "Διαγραφή", "Writen Off")
     .otherwise(F.col("e.ExposureStatus")).alias("Account_Status"),

    # OCW: CDI match first, TaxReg fallback
    F.coalesce(F.col("ocw_cdi.OCWStatus"), F.col("ocw_tax.OCWStatus")).alias("New_OCW_Status"),
    F.coalesce(F.col("ocw_cdi.OCWSubmittedDate"), F.col("ocw_tax.OCWSubmittedDate")).cast("date")
     .alias("New_OCW_Submitted_Date"),
    F.coalesce(F.col("ocw_cdi.OCWStatusDate"), F.col("ocw_tax.OCWStatusDate")).cast("date")
     .alias("New_OCW_Current_Status_Date"),

    F.col("p.DeceasedFlag"),
    F.to_date(F.lit(run_date)).alias("LetterDateCreated"),

    # ControlArea (same logic as Wholesale)
    F.when(F.col("e.PortfolioName").isin("Orbit", "Mercury"), "Retail")
     .when((F.col("e.PortfolioName") == "Light") & (F.col("p.Segment") == "Wholesale_CP"), "Wholesale")
     .when((F.col("e.PortfolioName") == "Light") & (F.coalesce(F.col("p.Segment"), F.lit("")) != "Wholesale_CP"), "Retail")
     .otherwise("").alias("ControlArea"),

    F.to_date(F.lit(run_date)).alias("BusinessDate")

).distinct()


# --- Row count and Delta write ---
row_count = df_cocpool.count()
print(f"CoCPoolData rows: {row_count}")

target_table = f"`{env}-cdh-delivery`.cocpooldata.CocPoolDataAll"

(df_cocpool.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(target_table))
print(f"Overwrote {target_table} with {row_count} rows")

print("Done.")

In [0]:
source_table = f"`{env}-cdh-delivery`.cocpooldata.CocPoolDataAll"
eligible_table = f"`{env}-cdh-delivery`.cocpooldata.CocPoolDataEligible"

# Reads CocPoolDataAll and applies eligibility filters:
#   1. OpenFlag=1, case_relation_type<>'Other', Queue NOT IN (QueueParams),
#      DeceasedFlag=0/NULL, Letter_Type IN (DEK Greek text)
#   2. RePoint join on 'repoint_coc' temp view (built in Temp Views cell):
#      Case_Status='Open' + active C-codes, excl. 'C6 Sent' → ~2k rows, matches old QC scale

# QueueParams exclusion list (on-prem: [PRC].[ssrsTmp].[coc_QueueParams])
queue_exclusions = [
    'Αναστολή Όχλησης N4469',
    'Special Treatment',
    'Αναστολή όχλησης by Asset Owner',
    'OCW_No_Treatment',
    'MRC_L3869_Pos_Future_Commencement_Day',
    'MRC_L3869_Pos_No Payment',
    'Up to 6 months suspension of Installment',
    'Mercury Ισχυρισμός Οφειλέτη Εξόφλησης',
    'Affected-Up to 6 months suspension of Installment',
    'Αναστολή Όχλησης_AML',
    'MRC_almost paid off',
    'Settlement L3869 external do not call interal',
    'Legal Paid Off Cases',
    'Αναστολή Όχλησης All Portfolios',
    'OCW 4738 No Treatment',
    'Gefyra account dimosiou',
    'Fulfilled_Settlement',
    'L3869 Pos Commen Day',
    'L3869 Pos No Payment',
    'Ισχυρισμός_Εξόφλησης',
    'Orbit Slaves Account',
    'Orbit WDN',
    'Orbit_slaves without Master',
    'Paid Off Prior Migration',
    'Orb orhans_no_pledge',
    'N.3869_Εξοφλήσεις',
    'N3869_For Default',
    'N3869_Εξαίρεση_διαχείρισης_Παράπονα',
    'Orb Externals special treatment',
    'N.3869_Έναρξη_Διαδικασίας_Έκπτωσης',
]

# Letter_Type filter: QCA Greek text values — pre-filter to DEK letter accounts (all 3 types)
# This scopes to the ~146k QCA DEK population; the RePoint join below then narrows to active cycle.
eligible_letter_codes = ['Αποστολή 1ης επιστολής ΔΕΚ', 'Αποστολή 2ης επιστολής ΔΕΚ', 'Αποστολή 3ης επιστολής ΔΕΚ']

# RePoint active cases — built as temp view 'repoint_coc' in Temp Views cell
df_repoint_active = spark.table("repoint_coc")

df_all = spark.table(source_table).filter(F.col("BusinessDate") == run_date)

df_eligible = (df_all
    .filter(F.col("CurrentlyOpenFlag").cast("boolean") == True)               # OpenFlag = 1
    .filter(F.col("case_relation_type") != 'Other')                           # case_relation_type <> 'Other'
    .filter(~F.col("Queue").isin(queue_exclusions))                            # Queue NOT IN (QueueParams)
    .filter(~F.coalesce(F.col("DeceasedFlag").cast("boolean"), F.lit(False)))  # ISNULL(DeceasedFlag,0) = 0
    .filter(F.col("Letter_Type").isin(eligible_letter_codes))                  # Letter_Type IN DEK text codes
    .join(F.broadcast(df_repoint_active),                                      # RePoint: active C-code filter
          F.col("CaseCode") == F.col("repoint_case_code"),
          "inner")
    .drop("repoint_case_code")
)

row_count = df_eligible.count()
print(f"CoCPoolDataEligible rows: {row_count}")

(df_eligible.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(eligible_table))
print(f"Overwrote {eligible_table} with {row_count} rows")

print("Done.")

In [0]:
import os
import glob
import shutil

output_dir = report_output
os.makedirs(output_dir, exist_ok=True)

date_suffix = run_date.replace("-", "_")

# Read both tables for the current BusinessDate
df_all = spark.table(f"`{env}-cdh-delivery`.cocpooldata.CocPoolDataAll").filter(F.col("BusinessDate") == run_date)
df_eligible = spark.table(f"`{env}-cdh-delivery`.cocpooldata.CocPoolDataEligible").filter(F.col("BusinessDate") == run_date)


def export_csv(sdf, output_dir, filename):
    """Write a Spark DataFrame as a single CSV file (semicolon-delimited, UTF-8 BOM for Excel)."""
    temp_dir = f"{output_dir}/_temp_{filename}"
    final_path = f"{output_dir}/{filename}"
    if os.path.exists(final_path):
        os.remove(final_path)
    (sdf.coalesce(1).write.mode("overwrite")
        .option("header", "true")
        .option("sep", ";")
        .option("quote", '"')
        .option("escape", '"')
        .csv(temp_dir))
    part_file = glob.glob(f"{temp_dir}/part-*.csv")[0]
    # Prepend UTF-8 BOM so Excel recognises Greek characters
    with open(part_file, "rb") as f:
        content = f.read()
    with open(final_path, "wb") as f:
        f.write(b"\xef\xbb\xbf" + content)
    shutil.rmtree(temp_dir)
    return final_path


all_path = export_csv(df_all, output_dir, f"coc_pool_data_all_{date_suffix}.csv")
print(f"CocPoolDataAll: {df_all.count():,} rows -> {all_path}")

eligible_path = export_csv(df_eligible, output_dir, f"coc_pool_data_eligible_{date_suffix}.csv")
print(f"CocPoolDataEligible: {df_eligible.count():,} rows -> {eligible_path}")

print("Done.")